# מפגש 11 — פתרון מורה מלא ובדיקות
כל תאי הקוד ניתנים להרצה על מקור הכיתה ועל נתוני העסקאות המומצאים; נדרשת בדיקת Run All בממשק Colab לפני אישור סופי.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
lab_df=pd.DataFrame({'value':[10,12,14,16,30]})
q1=lab_df['value'].quantile(.25,interpolation='linear')
q2=lab_df['value'].median()
q3=lab_df['value'].quantile(.75,interpolation='linear')
iqr=q3-q1
lower=q1-1.5*iqr
upper=q3+1.5*iqr
print('Five observations:',q1,q2,q3,iqr,lower,upper)
assert (q1,q2,q3,iqr,lower,upper)==(12.,14.,16.,4.,6.,22.)

## שפמים אינם גדרות — 10 ו־16, ו־30 נקודה לבדיקה

In [ ]:
valid=lab_df['value'][(lab_df['value']>=lower)&(lab_df['value']<=upper)]
flagged=lab_df['value'][(lab_df['value']<lower)|(lab_df['value']>upper)]
assert (valid.min(),valid.max())==(10,16)
assert flagged.tolist()==[30]
plt.figure(figsize=(6,4))
sns.boxplot(data=lab_df,y='value',whis=1.5,color='lightblue')
plt.title('Fences are not whisker endpoints')
plt.ylabel('Illustrative measurement')
plt.show()

## נתוני אמת — אורך סנפיר במינים שונים, לפי Pandas linear

In [ ]:
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
penguins_df=pd.read_csv(url)
order=['Adelie','Chinstrap','Gentoo']
plt.figure(figsize=(7,4))
sns.boxplot(data=penguins_df,x='species',y='flipper_length_mm',order=order,whis=1.5,color='lightblue')
plt.xlabel('Species')
plt.ylabel('Flipper length (mm)')
plt.title('Penguins: flipper lengths by species')
plt.show()
assert penguins_df.shape==(344,7)
assert penguins_df['flipper_length_mm'].count()==342

## חישוב קבוצתי מבוקר — למורה בלבד, לא למצגת לפני המעבדה

In [ ]:
stats={}
for species in order:
    series=penguins_df.loc[penguins_df['species']==species,'flipper_length_mm'].dropna()
    a=series.quantile(.25,interpolation='linear')
    b=series.quantile(.75,interpolation='linear')
    low=a-1.5*(b-a)
    high=b+1.5*(b-a)
    good=series[(series>=low)&(series<=high)]
    bad=series[(series<low)|(series>high)]
    stats[species]=(len(series),a,series.median(),b,good.min(),good.max(),len(bad))
print(stats)
assert stats['Adelie']==(151,186.,190.,195.,174.,208.,2)
assert stats['Chinstrap']==(68,191.,196.,201.,178.,212.,0)
assert stats['Gentoo']==(123,212.,216.,221.,203.,231.,0)

## מעבדת עסקאות — שבע רשומות, אחת כפולה לחלוטין
סימוני duplicated לפי מזהה לעומת price/items שונים; מסירים רק עותק ייבוא זהה, ורק אחרי אימות.

In [ ]:
raw=pd.DataFrame({'order_id':[201,202,203,203,204,205,206],'ordered_at':['2026-02-01','2026-02-02','2026-02-03','2026-02-03','2026-02-04','2026-02-05','2026-02-06'],'items':[1,2,3,3,2,1,2],'unit_price':[20.,10.,5.,5.,10.,20.,10.],'duration_min':[5.,9.,12.,12.,0.,-1.,np.nan]})
original=raw.copy(deep=True)
id_repeats=raw[raw.duplicated(subset=['order_id'],keep=False)]
measure_repeats=raw[raw.duplicated(subset=['items','unit_price'],keep=False)]
print('ID repeats',len(id_repeats),'Measurement repeats',len(measure_repeats))
assert len(raw)==7
assert len(id_repeats)==2
assert raw.duplicated().sum()==1
assert len(measure_repeats)>len(id_repeats)

## ניקוי בעותק בלבד, רק לאחר בדיקת הזהות
בהנחת ייבוא כפול של שורה זהה, drop_duplicates על **כל העמודות** אינו משליך בטעות על עסקאות דומות אחרות. חוזה משך חייב להיות >0.

In [ ]:
work=raw.copy(deep=True)
work=work.drop_duplicates(keep='first').copy()
assert len(work)==6
assert work['order_id'].is_unique
invalid=work['duration_min'].notna() & (work['duration_min']<=0)
print('New invalid durations:',int(invalid.sum()))
assert int(invalid.sum())==2
work.loc[invalid,'duration_min']=np.nan
print('Missing durations after:',int(work['duration_min'].isna().sum()))
assert int(work['duration_min'].isna().sum())==3
assert len(raw)==7 and raw.equals(original)
assert raw.loc[raw['order_id']==204,'duration_min'].iloc[0]==0
assert raw.loc[raw['order_id']==205,'duration_min'].iloc[0]==-1
assert work.loc[work['order_id']==204,'duration_min'].isna().all()
assert work.loc[work['order_id']==205,'duration_min'].isna().all()
display(work)

## יומן בקרה והגנה
לפני: 7 שורות, order_id 203 כפול באופן זהה, 1 משך חסר ו־2 לא־חיוביים. אחרי: 6 שורות, משך 204 ו־205 סומנו כ־NaN ויחד עם החסר המקורי מתקבלים 3 חסרים. לא נזרקו עסקאות עם כמות או מחיר זהים. אם אותו order_id כולל מידע סותר — עוצרים לבירור ולא keep-first שקט. אין להשתמש בתוצאות קבוצת Test לכוונון ניקוי.